# Explorare: instanțele, referințele și efectul schimbărilor (P2)

Notebook pentru etapele M1 și M2. Folosește instanțele de dezvoltare (`scripts/prepare_data.py`) sau, dacă îl aveți, pachetul disciplinei.

In [ ]:
import os, sys
from pathlib import Path
# notebook-ul se poate deschide din notebooks/ sau din rădăcină; lucrăm mereu din rădăcină
if not Path("config/default.yaml").exists() and Path("../config/default.yaml").exists():
    os.chdir("..")
sys.path.insert(0, "src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sada.config import load_config
cfg = load_config("config/default.yaml")
print("Lucrez în", Path.cwd())

In [ ]:
import glob, subprocess
from sada.p2.problem import Problem
files = sorted(glob.glob("data/p2_portofoliu/*.json")) or sorted(glob.glob("data/dev/dev_*.json"))
if not files:
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
    files = sorted(glob.glob("data/dev/dev_*.json"))
print("\n".join(files))
path = [f for f in files if "A_20" in f][0] if any("A_20" in f for f in files) else files[0]
p = Problem.from_json(path)
print(f"\n{p.name}: n = {p.n}, tip {p.kind}, buget {p.budget:,.0f}")

## 1. Cum arată problema

Dacă beneficiul crește aproape proporțional cu costul, strategia lacomă e greu de bătut. Dacă există intervenții ieftine cu beneficiu mare și altele scumpe cu beneficiu mic, ordinea contează mai mult.

In [ ]:
df = pd.DataFrame({"id": p.ids, "cost": p.cost, "benefit": p.benefit, "equity": p.equity})
df["raport"] = df.benefit / df.cost
display(df.sort_values("raport", ascending=False).head(10))
if p.kind == "binary":
    fig, ax = plt.subplots()
    sc = ax.scatter(df.cost, df.benefit, c=df.equity, cmap="viridis")
    fig.colorbar(sc, label="Echitate"); ax.set_xlabel("Cost"); ax.set_ylabel("Beneficiu")
    ax.set_title(f"{p.name}: cost total {df.cost.sum():,.0f}, buget {p.budget:,.0f}"); plt.show()

## 2. Referințele: optimul exact și strategia lacomă

In [ ]:
from sada.p2.baselines import greedy, random_search
from sada.p2.exact import solve_exact
if p.kind == "binary" and p.n <= 24:
    x_opt, v_opt = solve_exact(p)
    x_g, v_g, e_g = greedy(p)
    x_r, v_r, e_r = random_search(p, 4000, np.random.default_rng(0))
    print(f"optim exact {v_opt:.1f} | lacom {v_g:.1f} ({v_g / v_opt:.1%}) | aleator, 4000 evaluări {v_r:.1f} ({v_r / v_opt:.1%})")
else:
    print("Instanță întreagă sau prea mare pentru căutarea exhaustivă.")

## 3. Cât de mult mută o schimbare optimul?

Dacă optimul după o scădere a bugetului seamănă cu cel de dinainte, populația veche e un bun punct de pornire. Dacă nu, are nevoie de diversitate. Distanța Hamming între soluții vă spune care e cazul.

In [ ]:
if p.kind == "binary" and p.n <= 24:
    rows = []
    for f in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]:
        q = p.copy(); q.budget = p.budget * f
        x, v = solve_exact(q)
        rows.append({"factor_buget": f, "optim": v, "dist_hamming_fata_de_initial": int(np.abs(x - x_opt).sum())})
    res = pd.DataFrame(rows); display(res)
    res.plot(x="factor_buget", y="optim", marker="o", legend=False)
    plt.gca().invert_xaxis(); plt.ylabel("Valoarea optimă"); plt.title("Optimul în funcție de buget"); plt.show()

## Ce notați pentru nota de proiectare

- dimensiunea instanțelor și raportul buget / cost total;
- cât de aproape ajunge strategia lacomă de optim (referința de bătut);
- cât de diferit e optimul după fiecare tip de schimbare (alegerea răspunsului: hipermutație, imigranți, repornire).